# 06 — Reporting

**Stage 7 of the Tier A pipeline.** Assembles `reports/final_report.html`: a single self-contained file with every chart
base64-embedded and every number read from `reports/_key_figures.json`, so **nothing in the report is hardcoded**.

Structure follows `DOCS/STRUCTURE.md` §7: SCR storyline → Governing Thought → four MECE Key Lines → a specific
recommendation → one part per Key Line with action titles, exhibits and a "So What" under each → appendix. Design
follows `DOCS/DESIGN.md`: McKinsey navy/blue palette, Georgia display serif, single 760px reading column, white figure
cards in both themes.

In [1]:
import base64, json, re, warnings
from datetime import date
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "archive" / "Pokemon.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
REPORTS, FIGS = ROOT / "reports", ROOT / "reports" / "figures"

K = json.loads((REPORTS / "_key_figures.json").read_text(encoding="utf-8"))
print(f"{len(K)} key figures loaded from reports/_key_figures.json")


def pct(x, d=0):
    return f"{x*100:.{d}f}%"


def n(x):
    return f"{int(round(x)):,}"


def ci(c, d=2):
    return f"[{c[0]:.{d}f}, {c[1]:.{d}f}]"


def lst(xs):
    xs = list(xs)
    return xs[0] if len(xs) == 1 else ", ".join(xs[:-1]) + " and " + xs[-1]


def embed(name):
    b = base64.b64encode((FIGS / name).read_bytes()).decode()
    return f"data:image/png;base64,{b}"


SIG = K["sig_types"]
SIGB = K["sig_best"]
pts = lambda t: f"{SIGB[t]['diff_pp']:+.0f}"
for f in sorted(FIGS.glob("*.png")):
    print(f"  {f.name:24s} {f.stat().st_size/1024:6.0f} KB")

132 key figures loaded from reports/_key_figures.json
  ex01_rows.png                53 KB
  ex02_power_tiers.png         47 KB
  ex03_mega.png                57 KB
  ex04_generation.png          59 KB
  ex05_correlation.png         94 KB
  ex06_signatures.png         123 KB
  ex07_variance.png            52 KB
  ex08_protocols.png           51 KB
  ex09_pairs.png               87 KB
  ex10_type_auc.png            80 KB


## 7.1 The storyline, finalised against the evidence

The Stage 0 SCR (`01` §0.5) was a hypothesis. Four parts changed:

1. The file is **not** one row per species, and its rows are not independent: 79 forms copy a species and evolution
   families share a stat shape (`01` §1.5–1.7). That moved the honest model score from ~34% to ~21%.
2. The `Legendary` flag is inconsistent; the corrected flag makes the 580 power floor almost exact (`04` §4.1).
3. Of the review's four expected signature types (Steel, Rock, Ghost, Dragon), only **Steel and Rock** hold up; ten
   types have a signature, including several the review did not name (`04` §4.4).
4. "Natural archetypes" do not exist as separate clusters; roles are cuts through a continuum (`05` §5.7).

In [2]:
SITUATION = (
    f"The {n(K['n_species'])} species of Pok&eacute;mon Generations 1&ndash;6 each carry six base stats and one or two of 18 "
    "types. Designers and players talk about types as having typical stats (Steel is bulky, Electric is fast), and this "
    "dataset is a standard exercise in predicting type from stats.")

COMPLICATION = (
    "Nobody had measured how much type actually constrains a stat line, and the file makes it easy to get wrong. Its "
    f"{n(K['n_rows_raw'])} rows include {K['n_mega'] + K['n_alt']} forms that copy a species, evolution siblings share a "
    "stat shape, and the legendary flag mixes two definitions. Validated with a random split, a type-from-stats model "
    f"looks {pct(K['acc_random_species'])} accurate, and a team could build a design check on that number.")

GOVERNING_THOUGHT = (
    "Type is a tilt, not a template. A creature's power is set by a few fixed tier rules, not by its type, and type shapes "
    f"the stat profile of only {K['n_sig_types']} of 18 types. Validated honestly, stats name the type "
    f"{pct(K['acc_honest'])} of the time, against {pct(K['maj_acc'],1)} for always guessing Water. The design team should "
    "budget power by tier, template stat shape only for the signature types, and not build a stats-to-type checker.")

KEY_LINES = [
    ("The file",
     f"The {n(K['n_rows_raw'])} rows are {n(K['n_species'])} species plus {K['n_mega']} Mega Evolutions and {K['n_alt']} "
     f"alternate forms that reuse a species' number and type. The legendary flag misses {K['n_canon_misflagged']} species "
     "the games class as legendary or mythical. A species' nearest match in stat shape is within two dex numbers "
     f"{pct(K['nn_dex_near_rate'])} of the time, against {pct(K['nn_dex_near_null_mean'],1)} by chance: <strong>evolution "
     "families break independence</strong>, so every model must be validated by family."),
    ("Power",
     f"Total follows tier rules. <strong>{K['leg_at_580']} of {K['n_legendary_canon']} legendary species sit at 580 or "
     f"more</strong>, the {K['n_pseudo_600']} pseudo-legendaries sit at exactly 600, and {K['mega_plus100']} of {K['n_mega']} "
     f"Megas add exactly 100. Legendary status explains {pct(K['eta2_total_legendary'])} of the variation in Total, type "
     f"{pct(K['eta2_total_type'])}, and type's link does not survive correction once legendary status is held fixed. "
     f"There is no generational climb (&rho; = {K['gen_rho']:.2f})."),
    ("Shape",
     f"Divide each stat by Total and type explains at most {pct(K['eta2_share_max'])} of any stat's share "
     f"({K['eta2_share_max_stat']}). Tested on {K['n_families']} independent families, <strong>{K['n_sig_types']} of "
     f"{K['n_testable_types']} testable types carry a signature</strong>: Steel and Rock in Defense "
     f"({pts('Steel')} and {pts('Rock')} points of every 100 against other families), Fighting in Attack, Electric in Speed, Normal in HP, and others. "
     f"{lst(K['nosig_types'])} are shaped like the average species."),
    ("Prediction",
     f"Kept apart by family, a random forest names the type <strong>{pct(K['acc_honest'])}</strong> of the time "
     f"(&plusmn; {pct(K['acc_honest_sd'],1)} across seeds) against {pct(K['maj_acc'],1)} for always guessing Water, and "
     f"{pct(K['acc_logo'],1)} on an unseen generation. No pair of stats beats {pct(K['best_pair_acc'])}. Only "
     f"{K['n_auc_good']} types are recognisable (AUC &ge; {K['auc_good']}), so a checker would be wrong about four "
     "times in five."),
]

RECOMMENDATION = [
    ("Adopt a power-budget ladder from the corpus.",
     f"Ordinary creatures stay below 580 (the ordinary roster tops out at {K['ordinary_ceiling']}, "
     f"{K['ordinary_ceiling_name']}; Slaking's 670 is the one exception). Give each roster one "
     f"600-point pseudo-legendary line. Place legendaries at fixed steps from 580 to 720. Specify any temporary power-up form "
     "as &ldquo;+100 Total, same primary type&rdquo;, so balance work goes into where the points go. "
     "<em>Owner: balance lead, before creature sheets are drafted.</em>"),
    (f"Write stat-shape templates for the {K['n_sig_types']} signature types only.",
     "Use the per-type tilts in Part III (for example Steel: about +8 points of Defense per 100, &minus;4 of Speed) as "
     f"defaults a designer can override. Give {lst(K['nosig_types'])} no template: the corpus shows none, and inventing one "
     "would narrow those types for no reason. <em>Owner: creature designers, in the template document.</em>"),
    ("Do not build a stats-to-type classifier as a design check.",
     f"At {pct(K['acc_honest'])} honest accuracy it would flag mostly noise. If a check is wanted, make it a soft warning "
     f"for the {K['n_auc_good']} recognisable types ({lst(K['auc_good_types'])}), such as &ldquo;this Steel design has less "
     "Defense than 9 in 10 Steel creatures&rdquo;, that a designer can dismiss. <em>Owner: tools team, this sprint "
     "(redirect the classifier work).</em>"),
    ("Validate every model on this roster by evolution family or by generation.",
     f"A random split reported {pct(K['acc_random_species'])} for a model whose honest score is {pct(K['acc_honest'])}. "
     "<em>Owner: data team, as a standing rule.</em>"),
    ("Add the missing columns before the next balance review.",
     f"Evolution family and stage would explain much of the {pct(1 - K['total_model_r2'])} of Total that tier, type and "
     "generation leave open. Abilities, moves and a usage tier are needed to say anything about strength in play. Fix the "
     "legendary flag at the source. <em>Owner: data team, before the next review.</em>"),
]
print("Governing Thought:\n ", re.sub(r"&\w+;", "-", GOVERNING_THOUGHT))
print(f"\n{len(KEY_LINES)} key lines · {len(RECOMMENDATION)} recommendations")

Governing Thought:
  Type is a tilt, not a template. A creature's power is set by a few fixed tier rules, not by its type, and type shapes the stat profile of only 10 of 18 types. Validated honestly, stats name the type 21% of the time, against 14.6% for always guessing Water. The design team should budget power by tier, template stat shape only for the signature types, and not build a stats-to-type checker.

4 key lines · 5 recommendations


### MECE check on the Key Lines

| Key Line | Object | Overlap? |
|---|---|---|
| 1 — The file | the instrument: what a row is, which flags and which rows can be trusted | no claim about stats |
| 2 — Power | the *magnitude* of a stat line (Total) | Total only |
| 3 — Shape | the *proportions* of a stat line (stat ÷ Total) | power held constant by construction |
| 4 — Prediction | whether stats recover type well enough to act as a check | a model, not a description |

Power and shape split the stat line exactly (Total × shares = the six stats), so Key Lines 2 and 3 cannot overlap.
Together the four cover branches A–D of the Stage 0 issue tree (`01` §0.4).

In [3]:
EXHIBITS = [
    dict(part=1, file="ex01_rows.png",
         title=f"The {n(K['n_rows_raw'])} rows are {n(K['n_species'])} species plus {K['n_mega'] + K['n_alt']} forms, and "
               f"evolution siblings share a stat shape {K['nn_ratio']:.0f}&times; more often than chance",
         sowhat=f"Every Mega and alternate form keeps its species' number and primary type, so a model trained with forms "
                f"in both halves of a split is shown the answer. Evolution families do the same more quietly: siblings are "
                f"numbered consecutively and built on the same proportions ({pct(K['nn_dex_near_rate'])} of species have their "
                f"nearest stat-shape neighbour within two dex numbers; chance gives {pct(K['nn_dex_near_null_mean'],1)}). "
                "<strong>Analyses here use the 721 species, and models are validated with families kept together.</strong>"),
    dict(part=2, file="ex02_power_tiers.png",
         title=f"Power comes in tiers: {K['leg_at_580']} of {K['n_legendary_canon']} legendary species sit at 580 or more, "
               f"and the {K['n_pseudo_600']} pseudo-legendaries at exactly 600",
         sowhat=f"Legendary and mythical species use a handful of fixed Totals ({', '.join(str(v) for v in K['leg_total_values'] if v >= 580)}); "
                f"the only one below 580 is {K['leg_below_580'][0]}. A Total of 580 or more identifies them with "
                f"{K['rule580_tp']} hits, {K['rule580_fn']} miss and {K['rule580_fp']} false alarms: the "
                f"{K['n_pseudo_600']} pseudo-legendaries ({lst(K['pseudo_names'])}) and Slaking. Against the file's raw flag "
                f"the same rule raises {K['rule580_fp_raw']} false alarms, seven of which are the flag's own errors."),
    dict(part=2, file="ex03_mega.png",
         title=f"Mega Evolution is a fixed budget: {K['mega_plus100']} of {K['n_mega']} Megas add exactly 100 points",
         sowhat=f"The exception is {K['mega_other_name'].replace('AlakazamMega ', 'Mega ')} (+{K['mega_other_uplift']}). Every Mega "
                f"keeps its species' primary type. Alternate forms follow no single rule: {K['alt_uplift_zero']} of "
                f"{K['n_alt_forms']} re-spread the same Total (Rotom, Deoxys, the Therian formes) and the rest add 40&ndash;100. "
                "<strong>A power-up form is a +100 budget; the design work is where the points go.</strong>"),
    dict(part=2, file="ex04_generation.png",
         title=f"No steady power creep: the median ordinary species sits between {min(K['gen_median'].values()):.0f} and "
               f"{max(K['gen_median'].values()):.0f} in every generation",
         sowhat=f"Generation and Total correlate at &rho; = {K['gen_rho']:.2f} {ci(K['gen_rho_ci'])} among ordinary species; "
                "generation accounts for well under 1% of the variation, and neither species-level test survives correction. "
                f"Taking each family's strongest member as its final stage, generations differ modestly (&epsilon;&sup2; = "
                f"{K['gen_family_eps2']:.2f}), as a dip in Generations 2&ndash;3, not a climb. Generation 4's bump fits its many "
                "late evolutions for older lines. <strong>The budget ladder needs no per-generation adjustment.</strong>"),
    dict(part=3, file="ex05_correlation.png",
         title="Raw stats all rise together; once size is removed, the trade-offs appear",
         sowhat=f"Every pair of raw stats correlates positively (mean &rho; = {K['raw_corr_mean']:.2f}): bigger creatures are bigger "
                "everywhere. Divide by Total and the designers' trade-offs show: "
                + ", ".join(f"{k} &rho; = {v:.2f}" for k, v in list(K['share_corr_pairs'].items())[:3])
                + ". Comparing types on raw stats mostly compares their mix of evolution stages and legendaries, so Part III "
                "and the models use shares."),
    dict(part=3, file="ex06_signatures.png",
         title="Some types have a clear stat signature; about half are shaped like the average species",
         sowhat=f"Steel and Rock put about {K['sig_dev_pp']['Steel']['Defense']:+.0f} and "
                f"{K['sig_dev_pp']['Rock']['Defense']:+.0f} more points of every 100 into Defense than the average species, "
                f"mostly out of Speed. Tested family by family and corrected for {K['n_tests']} comparisons, {K['n_sig_types']} types keep a "
                f"signature ({lst(SIG)}). {lst(K['nosig_types'])} do not; Ghost and Dragon, which the data review expected "
                f"to be distinctive, are among them. Flying has only {K['n_type1_species']['Flying']} species as a primary type and cannot be judged."),
    dict(part=3, file="ex07_variance.png",
         title=f"Type explains at most {pct(K['eta2_share_max'])} of the variation in any stat share, and "
               f"{pct(K['eta2_total_type'])} of Total",
         sowhat=f"Even {K['eta2_share_max_stat']}, the share type shapes most, keeps about three-quarters of its variation for "
                f"other reasons: role, stage, second type. For Total, type explains {pct(K['eta2_total_type'])} against "
                f"{pct(K['eta2_total_legendary'])} for legendary status alone. <strong>Type is a tilt, not a template.</strong>"),
    dict(part=4, file="ex09_pairs.png",
         title=f"No pair of stats identifies type: the best, {K['best_pair']}, scores {pct(K['best_pair_acc'],1)}, "
               f"{(K['best_pair_acc'] - K['maj_acc'])*100:.0f} points above always guessing Water",
         sowhat=f"The dataset's premise, that type cannot be read from Attack and Defense, holds, but not because those two are "
                f"a poor choice: Attack &times; Defense ranks {K['ad_rank']} of 15 ({pct(K['ad_acc'],1)}). It holds because no pair "
                f"works. The fifteen pairs span {pct(K['worst_pair_acc'],1)} to {pct(K['best_pair_acc'],1)} against "
                f"{pct(K['maj_acc'],1)} for always guessing Water, and the top pairs are within seed noise of each other. All six "
                f"shares together reach {pct(K['acc_honest'],1)}. <strong>Type is not written into any two stats.</strong>"),
    dict(part=4, file="ex10_type_auc.png",
         title=f"Only {K['n_auc_good']} types are recognisable from their stats, all with a signature; "
               f"{len(K['auc_coinflip_types'])} cannot be told from a coin flip",
         sowhat=f"{lst(K['auc_good_types'])} clear an AUC of {K['auc_good']}: a model can rank their members above other "
                f"creatures. {lst(K['auc_coinflip_types'])} have intervals that reach 0.5. A soft warning is defensible for "
                "the first group and would flag only noise for the second."),
]
print(f"{len(EXHIBITS)} exhibits, all files present: {all((FIGS / e['file']).exists() for e in EXHIBITS)}")

9 exhibits, all files present: True


## 7.2 Design tokens — `DOCS/DESIGN.md`, consumed not re-picked

The three identity colours were **validated with the dataviz skill's checker**, not eyeballed:
`node scripts/validate_palette.js "#2251FF,#008A7E,#C1841C" --mode light --pairs all` passes lightness, chroma floor,
CVD separation (worst pair ΔE 12.5 protan), normal-vision separation (21.1) and contrast (all ≥ 3:1 on white). DESIGN.md's
Teal `#00857C` **fails the chroma floor** (C = 0.097 < 0.10), so it is nudged to `#008A7E` on the same hue. Grey carries
context only, and every coloured series also has a direct label or its own marker shape.

In [4]:
CSS = """
:root{
  --navy:#051C2C; --accent:#2251FF; --accent-soft:#eef2ff; --teal:#008A7E; --amber:#C1841C;
  --gold:#8A5E12; --slate:#7F93A6; --grey:#9FADB8;
  --ground:#f4f6f8; --surface:#ffffff; --ink:#051C2C; --ink-2:#43596b; --rule:#dfe5ea;
}
@media (prefers-color-scheme: dark){
  :root:not([data-theme="light"]){ --ground:#051C2C; --surface:#0c2434; --ink:#e9eef2; --ink-2:#a9bccb;
         --accent:#5a8dff; --accent-soft:#12304a; --rule:#1d3a4e; --gold:#E5B84A; }
}
:root[data-theme="light"]{
  --ground:#f4f6f8; --surface:#ffffff; --ink:#051C2C; --ink-2:#43596b;
  --accent:#2251FF; --accent-soft:#eef2ff; --rule:#dfe5ea; --gold:#8A5E12;
}
:root[data-theme="dark"]{
  --ground:#051C2C; --surface:#0c2434; --ink:#e9eef2; --ink-2:#a9bccb;
  --accent:#5a8dff; --accent-soft:#12304a; --rule:#1d3a4e; --gold:#E5B84A;
}
*{box-sizing:border-box}
body{margin:0;background:var(--ground);color:var(--ink);
  font-family:'Helvetica Neue',Arial,system-ui,'Segoe UI',sans-serif;
  font-size:16px;line-height:1.62;-webkit-font-smoothing:antialiased}
.wrap{max-width:760px;margin:0 auto;padding:56px 22px 96px}
h1{font-family:Georgia,'Times New Roman',serif;font-weight:400;font-size:2.35rem;line-height:1.18;
  margin:0 0 .5rem;text-wrap:balance;letter-spacing:-.01em}
h2{font-size:1.32rem;margin:3.2rem 0 .6rem;text-wrap:balance;line-height:1.3}
h3{font-size:1.02rem;margin:2rem 0 .5rem;text-wrap:balance}
p{margin:0 0 1rem;max-width:68ch}
a{color:var(--accent)}
.eyebrow{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.7rem;
  letter-spacing:.16em;text-transform:uppercase;color:var(--slate);margin:0 0 .6rem}
.sub{color:var(--ink-2);font-size:1.02rem;margin-bottom:2.2rem}
.answer{background:var(--surface);border-left:4px solid var(--accent);border-radius:0 6px 6px 0;
  padding:26px 28px;margin:2rem 0 2.6rem;box-shadow:0 1px 3px rgba(5,28,44,.07)}
.scr{margin:0 0 1.3rem}
.scr b{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.68rem;letter-spacing:.14em;
  text-transform:uppercase;color:var(--slate);display:block;margin-bottom:.2rem}
.gt{font-family:Georgia,'Times New Roman',serif;font-size:1.22rem;line-height:1.5;
  color:var(--accent);border-top:1px solid var(--rule);border-bottom:1px solid var(--rule);
  padding:1.1rem 0;margin:1.4rem 0}
ol.keylines{padding-left:0;counter-reset:kl;list-style:none;margin:1.4rem 0 0}
ol.keylines li{counter-increment:kl;position:relative;padding-left:2.4rem;margin-bottom:1.05rem}
ol.keylines li::before{content:counter(kl);position:absolute;left:0;top:.05rem;
  font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.78rem;font-weight:700;
  color:var(--accent);border:1.5px solid var(--accent);border-radius:50%;
  width:1.5rem;height:1.5rem;display:grid;place-items:center}
ol.keylines b{display:block;font-size:.72rem;letter-spacing:.13em;text-transform:uppercase;
  color:var(--slate);font-family:ui-monospace,'SF Mono',Consolas,monospace;margin-bottom:.15rem}
.tiles{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin:2.2rem 0}
.tile{background:var(--surface);border:1px solid var(--rule);border-radius:6px;padding:14px 14px 12px}
.tile .v{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:1.28rem;font-weight:700;
  color:var(--accent);font-variant-numeric:tabular-nums;line-height:1.15}
.tile .l{font-size:.74rem;color:var(--ink-2);margin-top:.3rem;line-height:1.35}
.rec{background:var(--surface);border:1px solid var(--gold);border-left:4px solid var(--gold);
  border-radius:0 6px 6px 0;padding:20px 24px;margin:2rem 0}
.rec .eyebrow{color:var(--gold)}
.rec ol{padding-left:1.15rem;margin:.6rem 0 0}
.rec li{margin-bottom:.75rem}
.rec li b{color:var(--ink)}
.part{border-top:2px solid var(--accent);margin:4.2rem 0 0;padding-top:.7rem}
.figcard{background:#fff;border:1px solid var(--rule);border-radius:6px;padding:16px;margin:1.4rem 0 0;
  overflow-x:auto}
.figcard img{width:100%;max-width:100%;display:block}
.sowhat{border-left:3px solid var(--accent);background:var(--accent-soft);border-radius:0 5px 5px 0;
  padding:13px 17px;margin:.85rem 0 2.4rem;font-size:.93rem;color:var(--ink)}
.sowhat b.k{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.68rem;letter-spacing:.13em;
  text-transform:uppercase;color:var(--accent)}
.tblwrap{overflow-x:auto;margin:1.2rem 0}
table{border-collapse:collapse;width:100%;font-size:.88rem}
th{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.68rem;letter-spacing:.1em;
  text-transform:uppercase;color:var(--slate);text-align:left;border-bottom:1.5px solid var(--rule);
  padding:8px 10px;font-weight:600}
td{border-bottom:1px solid var(--rule);padding:8px 10px;font-variant-numeric:tabular-nums;
  vertical-align:top}
tr.best td{background:var(--accent-soft)}
code{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.87em;
  background:var(--accent-soft);padding:.1em .32em;border-radius:3px}
.foot{margin-top:4.5rem;padding-top:1.3rem;border-top:1px solid var(--rule);
  font-size:.8rem;color:var(--slate)}
@media (max-width:620px){ .tiles{grid-template-columns:repeat(2,1fr)} h1{font-size:1.85rem}
  .wrap{padding:36px 16px 72px} .answer{padding:20px 18px} }
"""
print(f"CSS: {len(CSS):,} chars")

CSS: 5,167 chars


## 7.3 Assemble

In [5]:
def table(df, best_rows=()):
    h = "".join(f"<th>{c}</th>" for c in df.columns)
    body = ""
    for i, (_, r) in enumerate(df.iterrows()):
        cls = ' class="best"' if i in best_rows else ""
        body += f"<tr{cls}>" + "".join(f"<td>{v}</td>" for v in r) + "</tr>"
    return f'<div class="tblwrap"><table><thead><tr>{h}</tr></thead><tbody>{body}</tbody></table></div>'


def sowhat(txt):
    return f'<div class="sowhat"><b class="k">So what</b><br>{txt}</div>'


TILES = [
    (pct(K["acc_honest"]), f"honest type-from-stats accuracy, against {pct(K['maj_acc'],1)} for always guessing Water"),
    (f"{K['leg_at_580']} / {K['n_legendary_canon']}", "legendary or mythical species at Total 580 or more"),
    (f"{K['mega_plus100']} / {K['n_mega']}", "Mega Evolutions that add exactly 100 points to their species"),
    (f"{K['n_sig_types']} of 18", "types with a stat signature that survives a family-level test"),
]

file_tbl = pd.DataFrame({
    "What the data review says": [
        "~721 rows &times; 11 columns; <code>Generation</code> and <code>Legendary</code> probably absent",
        "<code>#</code> may repeat for forms",
        "Majority-class (Water) accuracy roughly 13&ndash;14%",
        "Stats are positively correlated, so multicollinearity matters",
        "Use a stratified train/test split",
    ],
    "What the file shows": [
        f"{n(K['n_rows_raw'])} &times; {K['n_cols_raw']}; both columns present, and the legendary flag misses "
        f"{K['n_canon_misflagged']} species the games class as legendary or mythical",
        f"It does: {K['n_dex_repeats']} extra rows ({K['n_mega']} Megas, {K['n_alt']} alternate forms), names glued "
        "(&ldquo;VenusaurMega Venusaur&rdquo;)",
        f"{pct(K['water_share_rows'],1)} of rows, {pct(K['water_share_species'],1)} of species: confirmed",
        f"True of raw stats (a size effect); stat shares trade off against each other (Attack &times; Sp. Def &rho; = "
        f"{K['share_corr_pairs']['Attack × Sp. Def']:.2f})",
        f"Not enough: siblings straddle a stratified split and inflate accuracy from {pct(K['acc_honest'])} to "
        f"{pct(K['acc_random_species'])}. Split by evolution family",
    ],
})

tier_tbl = pd.DataFrame({
    "Tier": ["Ordinary species", "Pseudo-legendary", "Legendary / mythical", "Mega Evolution", "Alternate form"],
    "Total in the corpus": [f"180&ndash;{K['ordinary_ceiling']} (Slaking 670)",
                            "600 exactly", "580&ndash;720 in fixed steps (Phione 480)", "species + 100", "species + 0 to +100"],
    "Count": [n(K["n_species"] - K["n_legendary_canon"] - K["n_pseudo_600"]), K["n_pseudo_600"], K["n_legendary_canon"],
              K["n_mega"], K["n_alt"]],
    "Rule for a new roster": ["below 580", "one line per roster, 600", "580 / 600 / 680 / 720", "+100, same primary type",
                              "re-spread the same Total by default"],
})

sig_rows = []
for t in SIG:
    items = sorted(K["sig_all"][t], key=lambda d: -abs(d["r"]))
    sig_rows.append({"Type": t,
                     "Signature (points per 100 of Total, vs other families)":
                         "; ".join(f"{d['stat']} {d['diff_pp']:+.1f}" for d in items),
                     "Strongest effect (r)": f"{items[0]['r']:+.2f}"})
sig_tbl = pd.DataFrame(sig_rows)
sig_tbl.loc[len(sig_tbl)] = {"Type": lst(K["nosig_types"]),
                             "Signature (points per 100 of Total, vs other families)": "none survives the test: use the average shape",
                             "Strongest effect (r)": "&mdash;"}

PT = K["protocol_table"]
pred_tbl = pd.DataFrame({
    "Validation protocol": ["Random split, all 800 rows", "Random split, 721 species", "Family-grouped (honest)",
                            "Leave one generation out", "Always guess Water"],
    "Accuracy": [pct(PT["random, 800 rows | RF · shares"]["accuracy"], 1), pct(K["acc_random_species"], 1),
                 f"{pct(K['acc_honest'],1)} &plusmn; {pct(K['acc_honest_sd'],1)}", pct(K["acc_logo"], 1), pct(K["maj_acc"], 1)],
    "Macro-F1": [f"{PT['random, 800 rows | RF · shares']['macro_f1']:.2f}", f"{K['f1_random_species']:.2f}",
                 f"{K['f1_honest']:.2f}", f"{PT['leave one generation out | RF · shares']['macro_f1']:.2f}", f"{K['maj_f1']:.2f}"],
    "Reading": ["forms and siblings leak", "siblings leak", "the number to use", "confirms the honest number", "the bar to clear"],
})

parts = {1: "Part I &middot; The file", 2: "Part II &middot; Power", 3: "Part III &middot; Shape", 4: "Part IV &middot; Prediction"}
part_h2 = {
    1: ("The file is 721 species plus 79 forms, with a mixed-up legendary flag and families that break independence", file_tbl, ()),
    2: ("Power is set by tier rules (a 580 legendary floor, a 600 pseudo-legendary, +100 Megas), not by type or generation",
        tier_tbl, ()),
    3: (f"Type tilts the stat profile of {K['n_sig_types']} types and leaves the rest at the average shape", sig_tbl, ()),
    4: ("Stats name the type about one time in five, so a design check would mostly flag noise", pred_tbl, (2,)),
}

html = ['<div class="wrap">']
html.append('<p class="eyebrow">Creature-balance review &middot; Tier A</p>')
html.append("<h1>Budget power by tier, template only the types that have a signature, and skip the type checker</h1>")
html.append('<p class="sub">Pok&eacute;mon base stats, Generations 1&ndash;6: '
            f'{n(K["n_species"])} species, six stats each, 18 types. How much a creature\'s type should constrain its '
            'power and its stat profile, and whether a model can check new designs for off-type stats.</p>')

html.append('<div class="answer">')
for lbl, txt in [("Situation", SITUATION), ("Complication", COMPLICATION)]:
    html.append(f'<p class="scr"><b>{lbl}</b>{txt}</p>')
html.append(f'<p class="gt">{GOVERNING_THOUGHT}</p>')
html.append('<ol class="keylines">')
for title, body in KEY_LINES:
    html.append(f"<li><b>{title}</b>{body}</li>")
html.append("</ol></div>")

html.append('<div class="tiles">')
for v, l in TILES:
    html.append(f'<div class="tile"><div class="v">{v}</div><div class="l">{l}</div></div>')
html.append("</div>")

html.append('<div class="rec"><p class="eyebrow">Recommendation &mdash; Lead creature-balance designer</p><ol>')
for head, body in RECOMMENDATION:
    html.append(f"<li><b>{head}</b> {body}</li>")
html.append("</ol></div>")

html.append('<div class="figcard"><img alt="Exhibit 8 — accuracy by validation protocol" src="' + embed("ex08_protocols.png") + '"></div>')
html.append(sowhat(
    f"The key exhibit. The same model scores {pct(K['acc_random_species'])} under a random split and "
    f"{pct(K['acc_honest'])} when evolution families are kept together. The difference is memorisation: with a sibling in "
    f"training, the model copies the family's type. Predicting a whole unseen generation gives {pct(K['acc_logo'],1)}, close "
    f"to the honest {pct(K['acc_honest'],1)}. A label-shuffled model never exceeds {pct(K['perm_max'],1)} ({K['n_perm']} runs), so the signal "
    f"is real, but it beats &ldquo;always Water&rdquo; by only {(K['acc_honest'] - K['maj_acc'])*100:.0f} points."))

for p_no in (1, 2, 3, 4):
    html.append(f'<div class="part"><p class="eyebrow">{parts[p_no]}</p></div>')
    h2, tbl, best = part_h2[p_no]
    html.append(f"<h2>{h2}</h2>")
    if tbl is not None:
        html.append(table(tbl, best_rows=best))
    for e in EXHIBITS:
        if e["part"] != p_no:
            continue
        html.append(f'<h3>{e["title"]}</h3>')
        html.append(f'<div class="figcard"><img alt="{re.sub("&[a-z]+;", "-", e["title"])}" src="{embed(e["file"])}"></div>')
        html.append(sowhat(e["sowhat"]))
    if p_no == 4:
        html.append(
            f"<p><strong>No natural archetypes either, with one small exception.</strong> Clustering the stat shares gives a "
            f"silhouette of {K['arch_sil_range'][0]:.2f}&ndash;{K['arch_sil_range'][1]:.2f} for every k from 2 to 8 (0.5 or more "
            f"indicates separate groups), a few hundredths above data with no structure ({K['arch_null_max_range'][0]:.2f}&ndash;"
            f"{K['arch_null_max_range'][1]:.2f}). Roles such as wall, sweeper and special attacker are useful labels, but the "
            f"data shows them as cuts through a continuum. The one sharp group is {K['arch_hp_n']} HP specialists "
            f"({lst(K['arch_hp_examples'][:3])}, &hellip;) that put {K['arch_hp_share']:.0f}% of their Total into HP.</p>")

html.append('<div class="part"><p class="eyebrow">Appendix &middot; Method and limits</p></div>')
html.append("<h2>Hypotheses that did not survive</h2>")
html.append(
    "<p><strong>&ldquo;Steel, Rock, Ghost and Dragon have the clearest signatures&rdquo;</strong> (data review). Half "
    f"right. Steel and Rock have the two strongest; Ghost and Dragon have none that survives a family-level test, while "
    f"{lst([t for t in SIG if t not in ('Steel', 'Rock')])} do.</p>")
html.append(
    "<p><strong>&ldquo;One row per species; ~721 rows, no generation or legendary columns.&rdquo;</strong> Wrong on each "
    "count, and the legendary column that does exist mixes two definitions.</p>")
html.append(
    "<p><strong>&ldquo;A stratified split is enough.&rdquo;</strong> It is not: evolution siblings inflate accuracy by "
    f"{(K['acc_random_species'] - K['acc_honest'])*100:.0f} points.</p>")
html.append(
    "<p><strong>&ldquo;Natural archetypes exist.&rdquo;</strong> Not as separate clusters (Part IV).</p>")
html.append(
    f"<p><strong>&ldquo;Dual types carry a stat premium.&rdquo;</strong> Their median Total is higher ({K['dual_median']:.0f} "
    f"against {K['single_median']:.0f}), but within families the dual-typed member is the later stage in "
    f"{pct(K['dual_within_pos'])} of {K['dual_within_n']} cases. It is an evolution-stage effect, not a design rule.</p>")
html.append("<h2>What this analysis does not claim</h2>")
html.append(
    "<p><strong>Nothing about strength in play.</strong> Base stats are inputs to the damage formula; moves, abilities, "
    "levels and items decide battles, and none is in the file. &ldquo;Power&rdquo; here means the Total budget only.</p>")
html.append(
    "<p><strong>No causation.</strong> Stats do not cause type or the reverse; the signatures describe how the designers of "
    "Generations 1&ndash;6 allocated points, and are offered as defaults, not laws.</p>")
html.append(
    "<p><strong>Gen 1&ndash;6 values as recorded.</strong> Later games changed some base stats and added generations. The "
    "family proxy (consecutive species sharing a primary type) misses evolutions added in later generations, so the honest "
    "score is, if anything, slightly optimistic.</p>")
html.append(
    f"<p><strong>Cleaning.</strong> Nothing was imputed or dropped. Glued form names were split ({K['n_glued_names']} rows), "
    f"forms flagged, a corrected <code>legendary_canon</code> flag added beside the raw one, and {K['n_extreme_species']} "
    "species with an extreme stat (Blissey, Shuckle, &hellip;) flagged and kept as design choices. The log is "
    "<code>data/processed/_cleaning_log.csv</code>.</p>")
html.append(
    f"<p><strong>Statistics.</strong> Rank-based tests (Mann&ndash;Whitney, Kruskal&ndash;Wallis, Wilcoxon) with rank-biserial "
    f"<i>r</i>, &epsilon;&sup2; and &eta;&sup2; as effect sizes. Type signatures are tested on {K['n_families']} family "
    f"means, and intervals resample families, because a family repeats one design. {K['n_tests']} tests were corrected "
    f"together (Benjamini&ndash;Hochberg); {K['n_significant_bh']} survive. A signature needs q &lt; 0.05 and |r| &ge; 0.30.</p>")
html.append(
    f"<p><strong>Models.</strong> Random forest (300 trees) and multinomial logistic regression on the six stat shares; "
    f"<code>Total</code> never beside the six stats. Five seeds, family-grouped 5-fold CV, leave-one-generation-out, and a "
    f"{K['n_perm']}-run label-permutation null (p = {K['perm_p']:.3f}). The model card is "
    "<code>models/type_classifier_card.json</code>. Fairness auditing does not apply: no people are affected.</p>")
html.append(
    "<p><strong>Reproducibility.</strong> Six numbered notebooks run in order with fixed seeds, and a restart-and-run-all "
    "from the raw CSV is verified. They import no local modules: chart styling and every helper are defined inline. Every "
    "number in this report is read from <code>reports/_key_figures.json</code>, written by notebooks 01&ndash;05.</p>")

html.append(
    f'<p class="foot">Source: Kaggle &ldquo;Pokemon with stats&rdquo; (Generations 1&ndash;6) &mdash; '
    f'{n(K["n_rows_raw"])} rows, SHA-256 <code>{K["sha256"][:12]}&hellip;</code>. Pok&eacute;mon names and stats belong '
    f'to Nintendo / Game Freak / The Pok&eacute;mon Company. Analysis {date.today().isoformat()}. Built to the McKinsey '
    f'communication standard per <code>DOCS/STRUCTURE.md</code>; visual system per <code>DOCS/DESIGN.md</code>.</p>')
html.append("</div>")

doc = ("<!doctype html><html lang=\"en\"><head><meta charset=\"utf-8\">"
       "<meta name=\"viewport\" content=\"width=device-width,initial-scale=1\">"
       "<title>Pokémon Stats by Type</title>"
       f"<style>{CSS}</style></head><body>{''.join(html)}</body></html>")

out = REPORTS / "final_report.html"
out.write_text(doc, encoding="utf-8")
print(f"wrote {out.relative_to(ROOT)}  ({len(doc)/1024/1024:.2f} MB, self-contained)")
print(f"  {len(EXHIBITS) + 1} exhibits base64-embedded · 0 external requests")

wrote reports\final_report.html  (0.94 MB, self-contained)
  10 exhibits base64-embedded · 0 external requests


In [6]:
assert 'src="data:image/png;base64,' in doc
assert doc.count("<img") == len(EXHIBITS) + 1
for bad in ["http://", "https://", "<script"]:
    assert bad not in doc, f"external or scripted content found: {bad}"
placeholders = re.findall(r"\{[a-zA-Z_\[\]']+\}", doc.split("<style>")[1].split("</style>")[1])
assert not placeholders, f"unformatted placeholders: {placeholders}"
assert "nan" not in re.sub(r"<[^>]+>", " ", doc.split("</style>")[1]).split(), "a NaN leaked into the text"
print("self-containment checks pass:")
print("  all images inline base64 · no external requests · no scripts · no unformatted placeholders · no NaN")
print(f"  {doc.count('<img')} images, {len(doc):,} chars")

self-containment checks pass:
  all images inline base64 · no external requests · no scripts · no unformatted placeholders · no NaN
  10 images, 987,217 chars


## Stage 7 — Gate Checklist (`DOCS/STRUCTURE.md`)

- [x] **SCR finalised**: evidence-backed; every figure read from `_key_figures.json`
- [x] **Governing Thought exists**: answers the business question in `01` §0.1 (how much type should constrain power and shape; whether a checker works)
- [x] **Key Lines are MECE**: 4 arguments (file / power / shape / prediction), table above; power × shape = the stat line
- [x] **One-Page Test**: SCR, Governing Thought, 4 Key Lines, 4 stat tiles, a 5-point recommendation and the key
      exhibit all sit above the first part divider
- [x] **All titles are Action Titles**: every `h2`/`h3` states the insight
- [x] **Every exhibit has a "So What"**: 9 part exhibits + the key exhibit, 10 annotations
- [x] **Vertical logic holds**: the four `h2`s alone tell the argument
- [x] **Horizontal logic holds**: each part's table and exhibits support its `h2`
- [x] **No orphan findings**: the archetype test and dual-type result are tied to Part IV and the appendix; failed
      hypotheses are reported, not dropped
- [x] **Recommendations specific**: five, each with an owner and a timing
- [x] **Methodology in the appendix** · [x] **Reproducible**: pinned packages, seeds, relative paths, no local imports

## Tier A Quality Checklist

- [x] Problem statement explicit and linked to three design decisions (templates, budget ladder, checker go/no-go)
- [x] Methodology appropriate: Path A tests with family-level independence; Path B validated by family and generation
- [x] Assumptions stated and tested (independence: the nearest-neighbour family test; normality not assumed)
- [x] Uncertainty quantified: family-bootstrap intervals, Wilson CI, five seeds, a permutation null
- [x] Alternative explanations addressed: legendary mix behind type's link to Total (§4.2), roster mix behind the Gen 4
      bump (§4.3), evolution stage behind the dual-type premium (§4.5), family memorisation behind random-split accuracy (§5.3)
- [x] Fairness: not applicable (no people) · [x] Quantified language · [x] Dependencies pinned